In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

zip_path = "DravidianCodeMix-2020.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset extracted")

In [ ]:
import os
os.listdir("dataset/DravidianCodeMix")

In [ ]:
import pandas as pd

train = pd.read_csv("dataset/DravidianCodeMix/tamil_offensive_full_train.csv", sep='\t')

print(train.shape)
train.head()

In [ ]:
train = train[['movie vara level la Erika poguthu', 'Not_offensive']]
train.columns = ['text', 'label']

train.head()

In [ ]:
train = train[train['label'] != 'not-Tamil']

train['label'].value_counts()

In [ ]:
def convert_label(label):
    if label == "Not_offensive":
        return 0
    else:
        return 1

train['label'] = train['label'].apply(convert_label)

train['label'].value_counts()

In [ ]:
train_path = "dataset/DravidianCodeMix/tamil_offensive_full_train.csv"
val_path   = "dataset/DravidianCodeMix/tamil_offensive_full_dev.csv"
test_path  = "dataset/DravidianCodeMix/tamil_offensive_full_test.csv"

In [ ]:
import pandas as pd

# Load
train = pd.read_csv(train_path, sep='\t')
val   = pd.read_csv(val_path, sep='\t')
test  = pd.read_csv(test_path, sep='\t')

# Keep only first 2 columns
train = train.iloc[:, :2]
val   = val.iloc[:, :2]
test  = test.iloc[:, :2]

train.columns = ['text', 'label']
val.columns   = ['text', 'label']
test.columns  = ['text', 'label']

train = train.dropna()
val   = val.dropna()
test  = test.dropna()

# Reset index
train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

In [ ]:
def convert_label(label):
    if str(label).strip() == "Not_offensive":
        return 0
    else:
        return 1

train['label'] = train['label'].apply(convert_label)
val['label']   = val['label'].apply(convert_label)
test['label']  = test['label'].apply(convert_label)

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    text = text.strip()

    return text

train['text'] = train['text'].apply(clean_text)

train.head()

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    text = text.strip()

    return text

train['text'] = train['text'].apply(clean_text)

train

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train['text'],
    train['label'],
    test_size=0.1,
    random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

In [ ]:
!pip install transformers
!pip install torch
!pip install datasets

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

print("Tokenizer loaded")

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete")

In [ ]:
import torch

class AbuseDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        item["labels"] = torch.tensor(self.labels.iloc[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = AbuseDataset(train_encodings, train_labels)
val_dataset = AbuseDataset(val_encodings, val_labels)

print("Datasets ready")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

print("Model loaded successfully")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(val_labels, preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, preds))

In [ ]:
test_dataset = AbuseDataset(test_encodings, test['label'])

In [ ]:
predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_scores = predictions.predictions[:, 1]

In [ ]:
from sklearn.metrics import roc_curve, auc

y_scores = predictions.predictions[:, 1]

fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - Tamil-EN (AUC = {roc_auc:.2f})")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_true, y_scores)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Tamil-EN")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

precision_val = precision_score(y_true, y_pred)
recall_val = recall_score(y_true, y_pred)
f1_val = f1_score(y_true, y_pred)
accuracy_val = accuracy_score(y_true, y_pred)

table = pd.DataFrame({
    "Language": ["Tamil-EN"],
    "Precision": [round(precision_val, 2)],
    "Recall": [round(recall_val, 2)],
    "F1-score": [round(f1_val, 2)],
    "Accuracy": [round(accuracy_val, 2)]
})

print(table)

In [ ]:
import pandas as pd
import numpy as np

predictions = trainer.predict(test_dataset)

# True labels
y_true = predictions.label_ids

# Predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

import torch.nn.functional as F
import torch

# Convert logits → probabilities
probs = F.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

# Take probability of Offensive class
y_scores = probs[:, 1]


# Create table
df_predictions = pd.DataFrame({
    "text": test['text'].values,
    "label": y_true,
    "predicted": y_pred,
    "prob_offensive": y_scores
})

df_predictions.head(10)

In [ ]:
import torch
import torch.nn.functional as F

def predict(text):

    # clean text
    text = clean_text(text)

    # tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # move tensors to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # model inference
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred].item()

    label = "Abusive" if pred == 1 else "Non-Abusive"

    print("Text:", text)
    print("Prediction:", label)
    print("Confidence:", round(confidence,3))

In [ ]:
predict("nee romba waste fellow")

In [ ]:
predict("super video bro nice work")

In [ ]:
model.save_pretrained("abuse_detection_model")
tokenizer.save_pretrained("abuse_detection_model")

In [ ]:
import shutil

shutil.make_archive("abuse_detection_model", 'zip', "abuse_detection_model")

In [ ]:
test_sentences = [

    # Non-abusive
    "super video bro nice work",
    "intha movie romba nalla iruku",
    "great acting by the hero",
    "bro unga explanation romba clear ah iruku",

    # Abusive
    "nee romba waste fellow",
    "ivan oru useless person",
    "dei nee enna loosu ah",
    "intha padam romba mokka",

    # Mixed examples
    "bro this trailer is amazing",
    "nee romba irritating bro",
    "intha actor acting super",
    "dei nee onnum use illa"

]

for sentence in test_sentences:
    print(" ------- ")
    predict(sentence)

In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

#  DATASET PATH
data_path = "dataset/DravidianCodeMix"

print("Available files:")
print(os.listdir(data_path))

# TAMIL FILES
train = pd.read_csv(f"{data_path}/tamil_offensive_full_train.csv", sep='\t')
val   = pd.read_csv(f"{data_path}/tamil_offensive_full_dev.csv", sep='\t')
test  = pd.read_csv(f"{data_path}/tamil_offensive_full_test.csv", sep='\t')

# FIX COLUMNS
train = train.iloc[:, :2]
val   = val.iloc[:, :2]
test  = test.iloc[:, :2]

train.columns = ['text', 'label']
val.columns   = ['text', 'label']
test.columns  = ['text', 'label']

# LABEL CONVERSION
def convert_label(label):
    return 0 if label == "Not_offensive" else 1

train['label'] = train['label'].apply(convert_label)
val['label']   = val['label'].apply(convert_label)
test['label']  = test['label'].apply(convert_label)

# CLEAN TEXT
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    return text.strip()

train['text'] = train['text'].apply(clean_text)
val['text']   = val['text'].apply(clean_text)
test['text']  = test['text'].apply(clean_text)


model = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model)

train_encodings = tokenizer(
    list(train['text']),
    truncation=True,
    padding="longest",
    max_length=256
)

val_encodings = tokenizer(
    list(val['text']),
    truncation=True,
    padding="longest",
    max_length=256
)

test_encodings = tokenizer(
    list(test['text']),
    truncation=True,
    padding="longest",
    max_length=256
)

# DATASET CLASS
class AbuseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = AbuseDataset(train_encodings, train['label'])
val_dataset   = AbuseDataset(val_encodings, val['label'])
test_dataset  = AbuseDataset(test_encodings, test['label'])

# CLASS WEIGHTS
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train['label']),
    y=train['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

# MODEL
model = AutoModelForSequenceClassification.from_pretrained(
    model,
    num_labels=2
)

# CUSTOM TRAINER
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# TRAINING CONFIG
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    logging_steps=100,
    save_total_limit=2,

    fp16=torch.cuda.is_available()
)

# TRAINER
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# TRAIN
trainer.train()

# VALIDATION METRICS
print("\n VALIDATION PERFORMANCE:")
val_metrics = trainer.evaluate()
print(val_metrics)

# FINAL TEST METRICS
print("\nTEST PERFORMANCE:")
test_metrics = trainer.evaluate(test_dataset)
print(test_metrics)

# SAVE BEST MODEL
trainer.save_model("tamil_abuse_model")
tokenizer.save_pretrained("tamil_abuse_model")

print("\nMODEL SAVED SUCCESSFULLY!")

In [ ]:
# Evaluation

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# CONFUSION MATRIX
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1], ["Non-Abusive", "Abusive"])
plt.yticks([0,1], ["Non-Abusive", "Abusive"])

plt.show()

# CLASSIFICATION REPORT
print("\n Classification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Non-Abusive", "Abusive"]
))

In [ ]:
from sklearn.metrics import roc_curve, auc

# Use same predictions object
y_scores = predictions.predictions[:, 1]

fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - Tamil-EN (AUC = {roc_auc:.2f})")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_true, y_scores)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Tamil-EN")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

precision_val = precision_score(y_true, y_pred)
recall_val = recall_score(y_true, y_pred)
f1_val = f1_score(y_true, y_pred)
accuracy_val = accuracy_score(y_true, y_pred)

table = pd.DataFrame({
    "Language": ["Tamil-EN"],
    "Precision": [round(precision_val, 2)],
    "Recall": [round(recall_val, 2)],
    "F1-score": [round(f1_val, 2)],
    "Accuracy": [round(accuracy_val, 2)]
})

print(table)

In [ ]:
import pandas as pd

df_examples = pd.DataFrame({
    "Text": test['text'][:10].values,
    "Actual": y_true[:10],
    "Predicted": y_pred[:10]
})

print(df_examples)

In [ ]:
import pandas as pd
import numpy as np

predictions = trainer.predict(test_dataset)

# True labels
y_true = predictions.label_ids

# Predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

import torch.nn.functional as F
import torch

# Convert logits → probabilities
probs = F.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

# Take probability of Offensive class
y_scores = probs[:, 1]


# Create table
df_predictions = pd.DataFrame({
    "text": test['text'].values,
    "label": y_true,
    "predicted": y_pred,
    "prob_offensive": y_scores
})

df_predictions.head(10)

In [ ]:
# LOAD SAVED MODEL
import torch
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "tamil_abuse_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

# SAME CLEANING (IMPORTANT)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    return text.strip()

# PREDICTION FUNCTION
def predict(text):
    cleaned = clean_text(text)

    inputs = tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred].item()

    label = "Abusive 🚫" if pred == 1 else "Non-Abusive ✅"

    print("\n Input:", text)
    print("Cleaned:", cleaned)
    print("Prediction:", label)
    print("Confidence:", round(confidence, 3))

    return label, confidence

In [ ]:
while True:
    user_text = input("\nEnter a sentence (or type 'exit'): ")

    if user_text.lower() == "exit":
        break

    predict(user_text)